# 03 · Climate Trend Analysis
**Climate Trend Analyzer** — step 3 of 5

### The statistical core
We want to answer: *is the climate actually changing, and how fast?*

Two independent methods, because one method can lie:

| Method | Type | Assumption | Strength |
| --- | --- | --- | --- |
| OLS with **monthly fixed effects** | parametric | linear trend, independent errors | gives a magnitude + CI |
| **Mann-Kendall** | non-parametric | monotonic trend only | robust to outliers, no distribution assumption |

### ⚠️ The pitfall everybody falls into
A monthly temperature series swings ±12 °C between summer and winter.
Regressing it on time **without** controlling for seasonality makes the
seasonal cycle leak into the trend estimate. The fix used here:

* **OLS** → `value ~ time + month dummies` (seasonal fixed effects)
* **Mann-Kendall** → run on the *deseasonalised* series

In [1]:
import sys, warnings
from pathlib import Path
warnings.filterwarnings("ignore")

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "src").exists():
    raise RuntimeError("Launch Jupyter from the project root or notebooks directory")
sys.path.insert(0, str(PROJECT_ROOT))

import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline
sns.set_theme(style="whitegrid", context="notebook", font_scale=1.1)

from src import config as cfg
from src import data_loader, preprocessing, features, eda, trend_analysis
from src import anomaly_detection, forecasting, insights, visualizations as viz

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 160)
print("Project root:", PROJECT_ROOT)

Project root: /home/user/Climate-Trend-Analyzer


In [2]:
feat = pd.read_csv(cfg.FEATURED_CSV, parse_dates=["date"])
feat.head()

,date,region,country,latitude,longitude,temp_mean_c,temp_min_c,temp_max_c,rainfall_mm,humidity_pct,co2_ppm,sea_level_mm,sim_heatwave,sim_drought,observed_temp_mean_c,data_quality_flag,year,month,quarter,season,...,rain_pct_of_normal,rain_anomaly_z,temp_mean_c_lag1,temp_mean_c_lag2,temp_mean_c_lag3,temp_mean_c_lag12,rainfall_mm_lag1,rainfall_mm_lag2,rainfall_mm_lag3,rainfall_mm_lag12,heatwave_month,drought_month,heatwave_event,drought_event,extreme_event,enso_proxy,co2_change_rate,sea_level_change_rate,month_sin,month_cos
0,1980-01-01,Hyderabad,India,17.4,78.5,26.94,21.41,32.27,32.8,61.5,338.58,-1.8,0,0,26.94,original,1980,1,1,Winter,...,200.707098,1.006636,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0,0,0,0,0,NaN,NaN,NaN,0.500000,8.660254e-01
1,1980-02-01,Hyderabad,India,17.4,78.5,30.22,25.69,34.08,26.8,53.1,339.69,-1.2,0,0,30.22,original,1980,2,1,Winter,...,116.285797,0.199628,26.94,NaN,NaN,NaN,32.8,NaN,NaN,NaN,0,0,0,0,0,-0.597067,NaN,NaN,0.866025,5.000000e-01
2,1980-03-01,Hyderabad,India,17.4,78.5,29.74,26.43,33.01,12.1,56.0,338.19,-0.9,0,0,29.74,original,1980,3,1,Spring,...,28.222671,-0.852250,30.22,26.94,NaN,NaN,26.8,32.8,NaN,NaN,0,1,0,0,0,-0.524015,NaN,NaN,1.000000,6.123234e-17
3,1980-04-01,Hyderabad,India,17.4,78.5,30.48,25.98,34.06,16.5,61.5,338.72,1.0,0,0,30.48,original,1980,4,2,Spring,...,20.077878,-1.048594,29.74,30.22,26.94,NaN,12.1,26.8,32.8,NaN,0,1,0,0,0,-0.706267,NaN,NaN,0.866025,-5.000000e-01
4,1980-05-01,Hyderabad,India,17.4,78.5,31.23,27.56,35.46,55.9,51.4,338.92,-0.4,0,0,31.23,repaired,1980,5,2,Spring,...,44.186618,-0.714401,30.48,29.74,30.22,NaN,16.5,12.1,26.8,NaN,0,0,0,0,0,-0.840178,NaN,NaN,0.500000,-8.660254e-01


## 3.1 Seasonal decomposition (STL)

In [3]:
decomp = trend_analysis.seasonal_decomposition(
    feat[feat.region == "London"].sort_values("date").set_index("date")["temp_mean_c"], period=12)
viz.plot_decomposition(decomp, "London")

[viz] saved 08_seasonal_decomposition.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/08_seasonal_decomposition.png')

The red line is the **trend** - it rises steadily. The green line is the
**seasonal** component (the annual cycle). The grey residuals are the noise
plus the extremes, which is exactly what the anomaly detector will look for.

## 3.2 Why we deseasonalise first

In [4]:
raw_series = feat[feat.region == "London"].sort_values("date")["temp_mean_c"]

print("Plain OLS on the raw series (WRONG - seasonality leaks in):")
from scipy.stats import linregress
plain = linregress(np.arange(len(raw_series)),raw_series)
print(f"   slope = {plain.slope*120:+.3f} C/decade,  R2 = {plain.rvalue**2:.4f}")

print()
print("OLS with monthly fixed effects (CORRECT):")
res2 = trend_analysis.ols_trend(raw_series, months=feat[feat.region=='London'].sort_values('date')['month'])
print(f"   slope = {res2['slope_per_year']*10:+.3f} C/decade,  R2 = {res2['r_squared']:.4f}")

Plain OLS on the raw series (WRONG - seasonality leaks in):
   slope = +0.413 C/decade,  R2 = 0.0100

OLS with monthly fixed effects (CORRECT):


   slope = +0.398 C/decade,  R2 = 0.9323


## 3.3 Mann-Kendall (implemented from scratch)

In [5]:
mk = trend_analysis.mann_kendall(raw_series, months=feat[feat.region=='London'].sort_values('date')['month'])
for k, v in mk.items():
    print(f"{k:28s}: {v}")

mk_trend                    : increasing
mk_s                        : 33706
mk_z                        : 8.0468276578933
mk_p_value                  : 8.49677004088232e-16
theil_sen_slope_per_year    : 0.037916323525325826


`mk_trend = increasing` and `theil_sen_slope_per_year` is the median of all
pairwise slopes - a *robust* alternative to least squares that is not
influenced by a few extreme months.

## 3.4 Trends for every region and indicator

In [6]:
trends = trend_analysis.analyze_trends(feat)
trends[trends.indicator == "temp_mean_c"][
    ["region","slope_per_decade","ci_low","ci_high","r_squared","p_value","mk_trend","mk_p_value"]
].round(4)

,region,slope_per_decade,ci_low,ci_high,r_squared,p_value,mk_trend,mk_p_value
0,Hyderabad,0.2687,0.0097,0.0441,0.8828,0.0022,increasing,0.0
5,London,0.3976,0.0231,0.0564,0.9323,0.0000,increasing,0.0
10,New York,0.3181,0.0153,0.0483,0.9743,0.0002,increasing,0.0
15,Singapore,0.2283,0.0049,0.0408,0.4277,0.0128,increasing,0.0
20,Sydney,0.2790,0.0119,0.0439,0.8991,0.0006,increasing,0.0


In [7]:
viz.plot_regional_warming_bar(trends)

[viz] saved 07_warming_rate_by_region.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/07_warming_rate_by_region.png')

## 3.5 Rainfall and CO₂

In [8]:
trends[trends.indicator.isin(["rainfall_mm","co2_ppm","sea_level_mm"])][
    ["region","indicator","slope_per_decade","r_squared","p_value","mk_trend"]
].round(4)

,region,indicator,slope_per_decade,r_squared,p_value,mk_trend
1,Hyderabad,rainfall_mm,-1.6085,0.2410,0.6235,no significant trend
2,Hyderabad,co2_ppm,28.2914,0.9951,0.0000,increasing
3,Hyderabad,sea_level_mm,31.3635,0.9831,0.0000,increasing
6,London,rainfall_mm,0.0517,0.1565,0.9769,no significant trend
7,London,co2_ppm,28.2340,0.9949,0.0000,increasing
8,London,sea_level_mm,31.4784,0.9829,0.0000,increasing
11,New York,rainfall_mm,0.2671,0.1142,0.9386,no significant trend
12,New York,co2_ppm,28.2407,0.9952,0.0000,increasing
13,New York,sea_level_mm,31.4781,0.9829,0.0000,increasing
16,Singapore,rainfall_mm,-12.9851,0.1874,0.0797,decreasing


In [9]:
viz.plot_co2_vs_temperature(feat)
viz.plot_sea_level_rise(feat)

[viz] saved 12_co2_vs_temperature.png


[viz] saved 13_sea_level_rise.png


PosixPath('/home/user/Climate-Trend-Analyzer/outputs/figures/13_sea_level_rise.png')

### ✅ Checkpoint
* All 5 regions show a **significant warming trend** (p < 0.05)
* OLS and Mann-Kendall agree on the direction
* CO₂ and sea level rise; rainfall shows no significant trend